# 04 · Train on the labelled data in full

Takes a model config written by notebook 02 or 03, and the cleaning run it names, so
the inputs and parameters are exactly those that were evaluated. Writes a self-contained
bundle to `outputs/model_<stamp>/`: model, vectorizer, token vocabulary, label hierarchy,
the cleaner, both configs, and the ID-to-name lookup. Notebook 05 needs nothing else.

NN only — how the final epoch is chosen:
- `"validation"`: a validation slice is held out; training stops early and the best epoch
  (kept on disk as a checkpoint) is restored.
- `"fixed_epochs"`: all rows are trained on for a set number of epochs, taken from
  `NN_FIXED_EPOCHS` or from a CV run's median best epoch.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))      # repo root, so `import classifier` works

DATA_DIR, CONFIG_DIR, OUTPUT_DIR = "../data", "../config", "../outputs"

import shutil
import pandas as pd

from classifier import io, plotting, crossval
from classifier.model import CategoryClassifier, ModelConfig, validation_split, hierarchy_lookup, \
    save_bundle, load_bundle

STAMP = io.stamp()
print("run stamp:", STAMP)

## Parameters

In [ ]:
MODEL_CONFIG_FILE = "model_YYYYMMDD_HHMMSS.json"   # in config/, from notebook 02 or 03
CLEANING_STAMP = None              # None = the cleaning run named in the model config
BLACKLIST_FILE = "from_config"     # "from_config" | None | a file in data/
EXCLUDE_FILE = None                # in data/: IDs to keep out (e.g. a test set). None = train on all

# ---- NN only ------------------------------------------------------------------------------
NN_TRAIN_MODE = "validation"       # "validation" | "fixed_epochs"
NN_VAL_FRAC = 0.05
NN_FIXED_EPOCHS = None             # an int, or None to take the median best epoch of CV_RUN
CV_RUN = None                      # e.g. "cv_20260930_150000"
SEED = 42

VERIFY_SAVE = True                 # reload the bundle and confirm identical predictions
NOTES = ""

## 1 · Config and data

In [ ]:
cfg = ModelConfig.load(f"{CONFIG_DIR}/{MODEL_CONFIG_FILE}")
CLEANING_STAMP = CLEANING_STAMP or cfg.cleaning_stamp
if CLEANING_STAMP != cfg.cleaning_stamp:
    print(f"WARNING: training on cleaning run {CLEANING_STAMP}, but this config was evaluated "
          f"on {cfg.cleaning_stamp}")
if BLACKLIST_FILE == "from_config":
    BLACKLIST_FILE = cfg.blacklist_file
cfg.cleaning_stamp, cfg.blacklist_file = CLEANING_STAMP, BLACKLIST_FILE
cfg.notes = (cfg.notes + " | " + NOTES).strip(" |")

cleaning_config, cleaner = io.load_cleaning(CONFIG_DIR, CLEANING_STAMP)
df = io.load_labelled(DATA_DIR, cleaning_config, cfg.id_col,
                      blacklist_file=BLACKLIST_FILE, exclude_file=EXCLUDE_FILE)
lookup = hierarchy_lookup(df, cfg.level_columns, cfg.name_columns)
print(f"model: {cfg.model_type} | config from {cfg.source} | cleaning {CLEANING_STAMP} | "
      f"blacklist {BLACKLIST_FILE}")

## 2 · Train

In [ ]:
MODEL_DIR = f"{OUTPUT_DIR}/model_{STAMP}"
train_df, val_df = df, None

if cfg.model_type == "nn":
    if NN_TRAIN_MODE == "validation":
        train_df, val_df = validation_split(df, NN_VAL_FRAC, cfg.level_columns[-1], SEED)
    elif NN_TRAIN_MODE == "fixed_epochs":
        epochs = NN_FIXED_EPOCHS or crossval.median_best_epoch(f"{OUTPUT_DIR}/{CV_RUN}")
        if not epochs:
            raise ValueError("set NN_FIXED_EPOCHS, or CV_RUN to a finished NN cross-validation run")
        cfg.nn = {**cfg.nn, "epochs": int(epochs)}
        print(f"training all {len(df):,} rows for {epochs} epochs")
    else:
        raise ValueError(f"NN_TRAIN_MODE={NN_TRAIN_MODE!r}")

clf = CategoryClassifier(cfg).fit(train_df, val_df, checkpoint_dir=f"{MODEL_DIR}/checkpoint")

In [ ]:
if clf.history is not None:
    fig_curves = plotting.plot_training_curves(clf.history, cfg.level_columns)

## 3 · Save the bundle

In [ ]:
save_bundle(MODEL_DIR, clf, cleaner, cleaning_config, lookup=lookup, manifest={
    "stamp": STAMP, "model_config_file": MODEL_CONFIG_FILE, "cleaning_stamp": CLEANING_STAMP,
    "blacklist_file": BLACKLIST_FILE, "exclude_file": EXCLUDE_FILE,
    "n_training_rows": len(train_df), "n_validation_rows": 0 if val_df is None else len(val_df),
    "nn_train_mode": NN_TRAIN_MODE if cfg.model_type == "nn" else None,
    "best_epoch": clf.history.attrs.get("best_epoch") if clf.history is not None else None,
})
shutil.rmtree(f"{MODEL_DIR}/checkpoint", ignore_errors=True)
if clf.history is not None:
    fig_curves.savefig(f"{MODEL_DIR}/training_curves.png", dpi=120, bbox_inches="tight")

for name in sorted(os.listdir(MODEL_DIR)):
    print(f"{name:<28} {os.path.getsize(f'{MODEL_DIR}/{name}') / 1e6:>9.1f} MB")
print(f"\nMODEL_DIR = \"{MODEL_DIR}\"   <- use this in notebook 05")

In [ ]:
if VERIFY_SAVE:
    sample = df[cfg.text_col].sample(min(500, len(df)), random_state=0)
    before = clf.predict(sample, verbose=False)
    after = load_bundle(MODEL_DIR)["clf"].predict(sample, verbose=False)
    same_path = (before.filter(like="Predicted").to_numpy() == after.filter(like="Predicted").to_numpy()).all()
    same_conf = ((before.filter(like="Confidence") - after.filter(like="Confidence")).abs().max().max() < 1e-5)
    print(f"identical paths: {same_path} | identical confidence: {same_conf}")
    assert same_path and same_conf, "the saved bundle does not reproduce the fitted model"